In [1]:
!uv add langchain-teddynote

Resolved 186 packages in 1ms
Checked 179 packages in 11ms


In [2]:
!uv add langchain-community

Resolved 186 packages in 1ms
Checked 179 packages in 4ms


In [ ]:
!uv add -U "langchain-text-splitters>=1.1.2" tiktoken beautifulsoup4

In [ ]:
!uv add -U langchain-experimental langchain-openai

In [6]:
from langchain_text_splitters import RecursiveJsonSplitter

data = {
    "factory": {
        "name": "서울 1공장",
        "lines": [
            {
                "line_id": "A",
                "machines": [
                    {"id": "A-01", "status": "running", "temperature": 42.1},
                    {"id": "A-02", "status": "warning", "temperautre": 78.4},
                ],
            },
            {
                "line_id": "B",
                "machines": [
                    {"id": "B-01", "status": "stopped", "temperature": 25.0}
                ],
            },
        ],
    }
}

splitter = RecursiveJsonSplitter(
    max_chunk_size = 180,
    min_chunk_size = 80,
)

json_chunks = splitter.split_json(
    json_data=data,
    convert_lists=True,
)

for index, chunk in enumerate(json_chunks, start=1):
    print(f"[JSON 청크 {index}]")
    print(chunk)

[JSON 청크 1]
{'factory': {'name': '서울 1공장', 'lines': {'0': {'line_id': 'A'}}}}
[JSON 청크 2]
{'factory': {'lines': {'0': {'machines': {'0': {'id': 'A-01', 'status': 'running', 'temperature': 42.1}, '1': {'id': 'A-02', 'status': 'warning', 'temperautre': 78.4}}}}}}
[JSON 청크 3]
{'factory': {'lines': {'1': {'line_id': 'B', 'machines': {'0': {'id': 'B-01', 'status': 'stopped', 'temperature': 25.0}}}}}}


In [7]:

documents = splitter.create_documents(
    texts=[data],
    metadatas=[{"source": "factory-status.json"}],
    convert_lists=True,
)

for document in documents:
    print(document.metadata)
    print(document.page_content)

{'source': 'factory-status.json'}
{"factory": {"name": "\uc11c\uc6b8 1\uacf5\uc7a5", "lines": {"0": {"line_id": "A"}}}}
{'source': 'factory-status.json'}
{"factory": {"lines": {"0": {"machines": {"0": {"id": "A-01", "status": "running", "temperature": 42.1}, "1": {"id": "A-02", "status": "warning", "temperautre": 78.4}}}}}}
{'source': 'factory-status.json'}
{"factory": {"lines": {"1": {"line_id": "B", "machines": {"0": {"id": "B-01", "status": "stopped", "temperature": 25.0}}}}}}


In [9]:
from langchain_text_splitters import CharacterTextSplitter

text = """스마트팩토리 데이터 파이프라인은 설비에서 발생한 데이터를 수집하고 저장하는 과정부터 시작합니다.
온도, 압력, 진동, 생산 수량과 같은 데이터는 PLC와 센서에서 생성되며 MES 또는 별도의 데이터 플랫폼으로 전달됩니다.

수집된 데이터는 그대로 분석에 사용하기보다 정제 과정이 필요합니다.
누락값, 비정상 값, 시간 형식 오류를 확인하고 분석 목적에 맞는 컬럼만 선택합니다.

정제된 데이터는 데이터베이스나 데이터 레이크에 저장됩니다.
데이터 엔지니어는 저장 위치, 파티션 구조, 스키마 변경 이력 등을 관리해야 합니다.

분석 단계에서는 생산량, 불량률, 설비 가동률, 공정능력 등의 지표를 계산할 수 있습니다.
분석 결과는 대시보드나 보고서를 통해 현업 사용자가 이해할 수 있는 형태로 제공됩니다.

RAG 시스템에서는 원본 문서를 한 번에 모델에 전달하지 않고 작은 청크로 분할합니다.
청크 크기가 지나치게 크면 검색 결과에 불필요한 문장이 많이 포함될 수 있습니다.

반대로 청크가 지나치게 작으면 질문에 필요한 문맥이 여러 조각으로 나뉠 수 있습니다.
따라서 문서의 구조와 질문 유형을 고려하여 적절한 chunk_size와 chunk_overlap을 설정해야 합니다."""

splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size = 20,
    chunk_overlap = 10,
    length_function = len,
    is_separator_regex=False,
)

chunks = splitter.split_text(text)

for index, chunk in enumerate(chunks, start=1):
    print(f"[청크 {index}] 길이={len(chunk)}")
    print(chunk)

Created a chunk of size 123, which is longer than the specified 20
Created a chunk of size 85, which is longer than the specified 20
Created a chunk of size 80, which is longer than the specified 20
Created a chunk of size 99, which is longer than the specified 20
Created a chunk of size 94, which is longer than the specified 20


[청크 1] 길이=123
스마트팩토리 데이터 파이프라인은 설비에서 발생한 데이터를 수집하고 저장하는 과정부터 시작합니다.
온도, 압력, 진동, 생산 수량과 같은 데이터는 PLC와 센서에서 생성되며 MES 또는 별도의 데이터 플랫폼으로 전달됩니다.
[청크 2] 길이=85
수집된 데이터는 그대로 분석에 사용하기보다 정제 과정이 필요합니다.
누락값, 비정상 값, 시간 형식 오류를 확인하고 분석 목적에 맞는 컬럼만 선택합니다.
[청크 3] 길이=80
정제된 데이터는 데이터베이스나 데이터 레이크에 저장됩니다.
데이터 엔지니어는 저장 위치, 파티션 구조, 스키마 변경 이력 등을 관리해야 합니다.
[청크 4] 길이=99
분석 단계에서는 생산량, 불량률, 설비 가동률, 공정능력 등의 지표를 계산할 수 있습니다.
분석 결과는 대시보드나 보고서를 통해 현업 사용자가 이해할 수 있는 형태로 제공됩니다.
[청크 5] 길이=94
RAG 시스템에서는 원본 문서를 한 번에 모델에 전달하지 않고 작은 청크로 분할합니다.
청크 크기가 지나치게 크면 검색 결과에 불필요한 문장이 많이 포함될 수 있습니다.
[청크 6] 길이=112
반대로 청크가 지나치게 작으면 질문에 필요한 문맥이 여러 조각으로 나뉠 수 있습니다.
따라서 문서의 구조와 질문 유형을 고려하여 적절한 chunk_size와 chunk_overlap을 설정해야 합니다.


In [12]:
from langchain_text_splitters import CharacterTextSplitter
from pathlib import Path
text = Path("data/paragraphs.txt").read_text(encoding="utf-8")

splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size = 30,
    chunk_overlap = 10,
    length_function = len,
    is_separator_regex=False,
)

chunks = splitter.split_text(text)

for index, chunk in enumerate(chunks, start=1):
    print(f"[청크 {index}] 길이={len(chunk)}")
    print(chunk)

Created a chunk of size 123, which is longer than the specified 30
Created a chunk of size 85, which is longer than the specified 30
Created a chunk of size 80, which is longer than the specified 30
Created a chunk of size 99, which is longer than the specified 30
Created a chunk of size 94, which is longer than the specified 30


[청크 1] 길이=123
스마트팩토리 데이터 파이프라인은 설비에서 발생한 데이터를 수집하고 저장하는 과정부터 시작합니다.
온도, 압력, 진동, 생산 수량과 같은 데이터는 PLC와 센서에서 생성되며 MES 또는 별도의 데이터 플랫폼으로 전달됩니다.
[청크 2] 길이=85
수집된 데이터는 그대로 분석에 사용하기보다 정제 과정이 필요합니다.
누락값, 비정상 값, 시간 형식 오류를 확인하고 분석 목적에 맞는 컬럼만 선택합니다.
[청크 3] 길이=80
정제된 데이터는 데이터베이스나 데이터 레이크에 저장됩니다.
데이터 엔지니어는 저장 위치, 파티션 구조, 스키마 변경 이력 등을 관리해야 합니다.
[청크 4] 길이=99
분석 단계에서는 생산량, 불량률, 설비 가동률, 공정능력 등의 지표를 계산할 수 있습니다.
분석 결과는 대시보드나 보고서를 통해 현업 사용자가 이해할 수 있는 형태로 제공됩니다.
[청크 5] 길이=94
RAG 시스템에서는 원본 문서를 한 번에 모델에 전달하지 않고 작은 청크로 분할합니다.
청크 크기가 지나치게 크면 검색 결과에 불필요한 문장이 많이 포함될 수 있습니다.
[청크 6] 길이=112
반대로 청크가 지나치게 작으면 질문에 필요한 문맥이 여러 조각으로 나뉠 수 있습니다.
따라서 문서의 구조와 질문 유형을 고려하여 적절한 chunk_size와 chunk_overlap을 설정해야 합니다.


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
text = Path("data/paragraphs.txt").read_text(encoding="utf-8")

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 60,
    chunk_overlap = 10,
    length_function = len,
    is_separator_regex=False,
)

chunks = splitter.split_text(text)

for index, chunk in enumerate(chunks, start=1):
    print(f"[청크 {index}] 길이={len(chunk)}")
    print(chunk)

[청크 1] 길이=53
스마트팩토리 데이터 파이프라인은 설비에서 발생한 데이터를 수집하고 저장하는 과정부터 시작합니다.
[청크 2] 길이=56
온도, 압력, 진동, 생산 수량과 같은 데이터는 PLC와 센서에서 생성되며 MES 또는 별도의 데이터
[청크 3] 길이=20
별도의 데이터 플랫폼으로 전달됩니다.
[청크 4] 길이=37
수집된 데이터는 그대로 분석에 사용하기보다 정제 과정이 필요합니다.
[청크 5] 길이=47
누락값, 비정상 값, 시간 형식 오류를 확인하고 분석 목적에 맞는 컬럼만 선택합니다.
[청크 6] 길이=32
정제된 데이터는 데이터베이스나 데이터 레이크에 저장됩니다.
[청크 7] 길이=47
데이터 엔지니어는 저장 위치, 파티션 구조, 스키마 변경 이력 등을 관리해야 합니다.
[청크 8] 길이=50
분석 단계에서는 생산량, 불량률, 설비 가동률, 공정능력 등의 지표를 계산할 수 있습니다.
[청크 9] 길이=48
분석 결과는 대시보드나 보고서를 통해 현업 사용자가 이해할 수 있는 형태로 제공됩니다.
[청크 10] 길이=48
RAG 시스템에서는 원본 문서를 한 번에 모델에 전달하지 않고 작은 청크로 분할합니다.
[청크 11] 길이=45
청크 크기가 지나치게 크면 검색 결과에 불필요한 문장이 많이 포함될 수 있습니다.
[청크 12] 길이=47
반대로 청크가 지나치게 작으면 질문에 필요한 문맥이 여러 조각으로 나뉠 수 있습니다.
[청크 13] 길이=59
따라서 문서의 구조와 질문 유형을 고려하여 적절한 chunk_size와 chunk_overlap을 설정해야
[청크 14] 길이=9
설정해야 합니다.


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text= """# 생산 설비 이상 감지 보고서

2026년 9월 조립 라인의 설비 데이터를 분석한 결과, 일부 구간에서 진동 증가와 온도 상승이 동시에 관찰되었습니다.
설비 상태는 정상, 주의, 경고 세 단계로 분류했습니다.

## 데이터 수집

설비별 센서 데이터는 10초 간격으로 저장되었습니다.
주요 컬럼은 timestamp, machine_id, temperature, vibration, pressure, status입니다.
일부 설비에서는 네트워크 지연으로 인해 30초 이상 데이터가 누락된 구간이 있었습니다.

데이터 전처리 과정에서는 중복 timestamp를 제거하고 결측 구간을 별도로 표시했습니다.
단순 평균 대체는 이상 탐지 결과를 왜곡할 수 있기 때문에 사용하지 않았습니다.

## 이상 징후

A-03 설비는 오전 10시 20분부터 진동값이 점진적으로 증가했습니다.
정상 구간의 평균 진동은 2.1 mm/s였지만 이상 구간에서는 최대 6.8 mm/s까지 상승했습니다.

같은 시간대에 베어링 온도도 54도에서 78도까지 상승했습니다.
진동과 온도가 동시에 증가한 점을 고려하면 단순 센서 오류보다는 실제 설비 상태 변화 가능성이 높습니다.

## 조치 결과

현장 점검 결과 A-03 설비의 베어링 윤활 상태가 기준보다 낮았습니다.
윤활 작업 후 진동은 2.4 mm/s 수준으로 회복되었고 온도도 57도까지 감소했습니다.

향후에는 진동과 온도를 결합한 경고 규칙을 적용할 예정입니다.
또한 단일 임계값 방식과 머신러닝 기반 이상 탐지 방식의 성능을 비교할 계획입니다."""

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 90,
    chunk_overlap = 15,
    separators = ["\n\n", "\n", "."," ",""],
    length_function = len,
    add_start_index=True,
)

documents = splitter.create_documents(
    [text],
    metadatas=[{"source": "guide.md"}],
)

for document in documents:
    print(document.metadata)
    print(document.page_content)

{'source': 'guide.md', 'start_index': 0}
# 생산 설비 이상 감지 보고서
{'source': 'guide.md', 'start_index': 19}
2026년 9월 조립 라인의 설비 데이터를 분석한 결과, 일부 구간에서 진동 증가와 온도 상승이 동시에 관찰되었습니다.
{'source': 'guide.md', 'start_index': 86}
설비 상태는 정상, 주의, 경고 세 단계로 분류했습니다.
{'source': 'guide.md', 'start_index': 119}
## 데이터 수집
{'source': 'guide.md', 'start_index': 130}
설비별 센서 데이터는 10초 간격으로 저장되었습니다.
{'source': 'guide.md', 'start_index': 160}
주요 컬럼은 timestamp, machine_id, temperature, vibration, pressure, status입니다.
{'source': 'guide.md', 'start_index': 235}
일부 설비에서는 네트워크 지연으로 인해 30초 이상 데이터가 누락된 구간이 있었습니다.
{'source': 'guide.md', 'start_index': 285}
데이터 전처리 과정에서는 중복 timestamp를 제거하고 결측 구간을 별도로 표시했습니다.
{'source': 'guide.md', 'start_index': 337}
단순 평균 대체는 이상 탐지 결과를 왜곡할 수 있기 때문에 사용하지 않았습니다.
{'source': 'guide.md', 'start_index': 383}
## 이상 징후
{'source': 'guide.md', 'start_index': 393}
A-03 설비는 오전 10시 20분부터 진동값이 점진적으로 증가했습니다.
{'source': 'guide.md', 'start_index': 434}
정상 구간의 평균 진동은 2.1 mm/s였지만 이상 구간에서는 최대 6.8 mm/s까지 상승했습니다.
{'s

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

text = Path("data/general_document.txt").read_text(encoding="utf-8")
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 120,
    chunk_overlap = 15,
    separators = ["\n\n", "\n", "."," ",""],
    length_function = len,
    add_start_index=True,
)

documents = splitter.create_documents(
    [text],
    metadatas=[{"source": "guide.md"}],
)

for document in documents:
    print(document.metadata)
    print(document.page_content)

{'source': 'guide.md', 'start_index': 0}
# 생산 설비 이상 감지 보고서

2026년 9월 조립 라인의 설비 데이터를 분석한 결과, 일부 구간에서 진동 증가와 온도 상승이 동시에 관찰되었습니다.
설비 상태는 정상, 주의, 경고 세 단계로 분류했습니다.
{'source': 'guide.md', 'start_index': 119}
## 데이터 수집
{'source': 'guide.md', 'start_index': 130}
설비별 센서 데이터는 10초 간격으로 저장되었습니다.
주요 컬럼은 timestamp, machine_id, temperature, vibration, pressure, status입니다.
{'source': 'guide.md', 'start_index': 235}
일부 설비에서는 네트워크 지연으로 인해 30초 이상 데이터가 누락된 구간이 있었습니다.
{'source': 'guide.md', 'start_index': 285}
데이터 전처리 과정에서는 중복 timestamp를 제거하고 결측 구간을 별도로 표시했습니다.
단순 평균 대체는 이상 탐지 결과를 왜곡할 수 있기 때문에 사용하지 않았습니다.

## 이상 징후
{'source': 'guide.md', 'start_index': 383}
## 이상 징후

A-03 설비는 오전 10시 20분부터 진동값이 점진적으로 증가했습니다.
정상 구간의 평균 진동은 2.1 mm/s였지만 이상 구간에서는 최대 6.8 mm/s까지 상승했습니다.
{'source': 'guide.md', 'start_index': 492}
같은 시간대에 베어링 온도도 54도에서 78도까지 상승했습니다.
진동과 온도가 동시에 증가한 점을 고려하면 단순 센서 오류보다는 실제 설비 상태 변화 가능성이 높습니다.

## 조치 결과
{'source': 'guide.md', 'start_index': 588}
## 조치 결과

현장 점검 결과 A-03 설비의 베어링 윤활 상태가 기준보다 낮았습니다.
윤활 작업

In [21]:
import tiktoken
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)
from pathlib import Path

text = Path("data/data_engineering_sample.txt").read_text(encoding="utf-8")

encoding = tiktoken.get_encoding("cl100k_base")
token_count= len(encoding.encode(text))
print(f"전체 토근 수 : {token_count}")

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size = 80,
    chunk_overlap = 5,
)

token_chunks = token_splitter.split_text(text)

for index, chunk in enumerate(token_chunks, start=1):
    size = len(encoding.encode(chunk))
    print(f"[TokenTextSplitter {index}] 토큰={size}")

전체 토근 수 : 606
[TokenTextSplitter 1] 토큰=80
[TokenTextSplitter 2] 토큰=80
[TokenTextSplitter 3] 토큰=80
[TokenTextSplitter 4] 토큰=80
[TokenTextSplitter 5] 토큰=80
[TokenTextSplitter 6] 토큰=80
[TokenTextSplitter 7] 토큰=80
[TokenTextSplitter 8] 토큰=80
[TokenTextSplitter 9] 토큰=6


In [22]:
safe_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=80,
    chunk_overlap=5,
)

safe_chunks = safe_splitter.split_text(text)

for index, chunk in enumerate(safe_chunks, start=1):
    print(index, len(encoding.encode(chunk)), repr(chunk))

1 16 '데이터 엔지니어링 학습 예제 문서'
2 78 '1. 데이터 수집\n데이터 엔지니어는 여러 시스템에서 데이터를 수집합니다.\n수집 대상에는 데이터베이스, API, 로그 파일, CSV 파일 등이 있습니다.\n수집된 데이터는 이후 처리와 분석을 위해 저장됩니다.'
3 68 '2. 데이터 전처리\n수집한 데이터에는 결측값, 중복값, 잘못된 형식이 포함될 수 있습니다.\n전처리 과정에서는 데이터를 정제하고 필요한 형식으로 변환합니다.'
4 17 'Python과 Pandas는 데이터 전처리에 자주 사용됩니다.'
5 60 '3. ETL 파이프라인\nETL은 Extract, Transform, Load의 약자입니다.\nExtract 단계에서는 원본 데이터를 가져옵니다.\nTransform 단계에서는 데이터를 정제하고 변환합니다.'
6 28 'Load 단계에서는 처리된 데이터를 데이터베이스나 데이터 웨어하우스에 저장합니다.'
7 60 '4. 데이터베이스\n관계형 데이터베이스에서는 SQL을 사용하여 데이터를 조회하고 수정합니다.\n대표적인 관계형 데이터베이스에는 MySQL, PostgreSQL, Oracle 등이 있습니다.'
8 31 '대용량 데이터 환경에서는 데이터 웨어하우스와 데이터 레이크도 자주 사용됩니다.'
9 72 '5. RAG와 문서 로딩\nRAG는 외부 문서를 검색한 뒤 검색 결과를 LLM의 답변 생성에 활용하는 방식입니다.\nLangChain의 TextLoader를 이용하면 txt 파일을 Document 객체로 불러올 수 있습니다.'
10 50 '불러온 문서는 필요에 따라 여러 조각으로 나누고 임베딩한 뒤 벡터 데이터베이스에 저장할 수 있습니다.'
11 60 '6. 간단한 질문 예시\n데이터 전처리에 자주 사용되는 Python 라이브러리는 무엇인가요?\nETL의 세 단계는 무엇인가요?'
12 65 'RAG에서 문서를 불러온 뒤 어떤 과정을 거칠 수 있나요?\n관계형 데이터베이스에서 데이터를 조회할 때 주로 사용하는 언어는 무엇인가요?'


In [25]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
from pathlib import Path

text = Path("data/semantic_topics.txt").read_text(encoding="utf-8")

splitter = SemanticChunker(
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=80,
)

documents= splitter.create_documents([text])

for index, document in enumerate(documents, start=1):
    print(f"[의미 청크 {index}]")
    print(document.page_content)

[의미 청크 1]
RAG 시스템은 외부 문서를 검색하여 언어 모델의 답변을 보강하는 방식입니다. 검색 품질은 문서 분할, 임베딩 모델, 벡터 검색 방식, 프롬프트 구성의 영향을 받습니다. 문서 청크가 질문의 의미를 잘 보존해야 검색 결과의 정확도가 높아집니다. 반도체 노광 공정에서는 웨이퍼 위에 미세한 회로 패턴을 형성합니다. 노광 장비는 광원, 렌즈, 마스크, 웨이퍼 스테이지 등 여러 정밀 시스템으로 구성됩니다.
[의미 청크 2]
공정 조건의 작은 변화도 선폭과 정렬 정확도에 영향을 줄 수 있습니다.
[의미 청크 3]
전기자동차 배터리는 셀, 모듈, 팩 구조로 구성될 수 있습니다. 조립 공정에서는 정렬, 압착, 용접, 검사 등 다양한 공정 조건을 관리합니다.
[의미 청크 4]
공정 데이터 분석을 통해 불량 패턴과 설비 이상을 조기에 확인할 수 있습니다. 축구 경기 데이터 분석에서는 선수의 패스, 슈팅, 이동 거리, 점유율과 같은 지표를 활용합니다. 이벤트 데이터와 위치 데이터를 결합하면 경기 흐름과 선수의 역할을 더 세밀하게 분석할 수 있습니다. 동일한 경기라도 분석 목적에 따라 필요한 데이터와 평가 지표가 달라집니다. 커피 추출에서는 원두 분쇄도, 물 온도, 추출 시간, 원두와 물의 비율이 맛에 영향을 줍니다. 분쇄도가 너무 고우면 과다 추출이 발생할 수 있고 너무 굵으면 맛이 약해질 수 있습니다. 일관된 추출을 위해서는 레시피와 측정 기준을 일정하게 유지하는 것이 중요합니다.


In [26]:
from langchain_text_splitters import(
    Language,
    RecursiveCharacterTextSplitter,
)

python_code = """from dataclasses import dataclass
from statistics import mean


@dataclass
class SensorReading:
    machine_id: str
    temperature: float
    vibration: float
    pressure: float


class MachineMonitor:
    def __init__(self, temp_limit=75.0, vibration_limit=5.0):
        self.temp_limit = temp_limit
        self.vibration_limit = vibration_limit
        self.history = []

    def add_reading(self, reading: SensorReading):
        self.history.append(reading)

    def check_status(self, reading: SensorReading):
        if (
            reading.temperature >= self.temp_limit
            and reading.vibration >= self.vibration_limit
        ):
            return "critical"
        if reading.temperature >= self.temp_limit:
            return "temperature_warning"
        if reading.vibration >= self.vibration_limit:
            return "vibration_warning"
        return "normal"

    def average_temperature(self):
        if not self.history:
            return None
        return mean(item.temperature for item in self.history)


def parse_sensor_row(row: dict) -> SensorReading:
    return SensorReading(
        machine_id=row["machine_id"],
        temperature=float(row["temperature"]),
        vibration=float(row["vibration"]),
        pressure=float(row["pressure"]),
    )


def create_alert_message(reading: SensorReading, status: str) -> str:
    return (
        f"machine={reading.machine_id}, "
        f"temperature={reading.temperature}, "
        f"vibration={reading.vibration}, "
        f"status={status}"
    )


def run_demo():
    raw_rows = [
        {"machine_id": "A-01", "temperature": 54.2, "vibration": 2.1, "pressure": 1.01},
        {"machine_id": "A-02", "temperature": 79.4, "vibration": 3.0, "pressure": 1.03},
        {"machine_id": "A-03", "temperature": 82.1, "vibration": 6.4, "pressure": 1.04},
    ]

    monitor = MachineMonitor()

    for row in raw_rows:
        reading = parse_sensor_row(row)
        monitor.add_reading(reading)
        status = monitor.check_status(reading)
        print(create_alert_message(reading, status))

    print("average_temperature:", monitor.average_temperature())


if __name__ == "__main__":
    run_demo()
"""

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size = 140,
    chunk_overlap = 20,
    add_start_index = True,
)

documents = splitter.create_documents(
    [python_code],
    metadatas=[{"source": "sensor_monitor.py", "language": "python"}],
)

for index, document in enumerate(documents, start=1):
    print(f"[코드 청크 {index}] {document.metadata}")
    print(document.page_content)

[코드 청크 1] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 0}
from dataclasses import dataclass
from statistics import mean


@dataclass
[코드 청크 2] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 75}
class SensorReading:
    machine_id: str
    temperature: float
    vibration: float
    pressure: float
[코드 청크 3] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 182}
class MachineMonitor:
    def __init__(self, temp_limit=75.0, vibration_limit=5.0):
        self.temp_limit = temp_limit
[코드 청크 4] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 311}
self.vibration_limit = vibration_limit
        self.history = []
[코드 청크 5] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 381}
def add_reading(self, reading: SensorReading):
        self.history.append(reading)
[코드 청크 6] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 470}
def check_status(self, reading: SensorReading):
 

In [28]:
from langchain_text_splitters import(
    Language,
    RecursiveCharacterTextSplitter,
)

python_code = Path("data/sensor_monitor.py").read_text(encoding="utf-8")

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size = 200,
    chunk_overlap = 20,
    add_start_index = True,
)

documents = splitter.create_documents(
    [python_code],
    metadatas=[{"source": "sensor_monitor.py", "language": "python"}],
)

for index, document in enumerate(documents, start=1):
    print(f"[코드 청크 {index}] {document.metadata}")
    print(document.page_content)

[코드 청크 1] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 0}
from dataclasses import dataclass
from statistics import mean


@dataclass
class SensorReading:
    machine_id: str
    temperature: float
    vibration: float
    pressure: float
[코드 청크 2] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 182}
class MachineMonitor:
    def __init__(self, temp_limit=75.0, vibration_limit=5.0):
        self.temp_limit = temp_limit
        self.vibration_limit = vibration_limit
        self.history = []
[코드 청크 3] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 381}
def add_reading(self, reading: SensorReading):
        self.history.append(reading)
[코드 청크 4] {'source': 'sensor_monitor.py', 'language': 'python', 'start_index': 470}
def check_status(self, reading: SensorReading):
        if (
            reading.temperature >= self.temp_limit
            and reading.vibration >= self.vibration_limit
        ):
[코드 청크 5] {'source': 'senso

In [29]:
separators = RecursiveCharacterTextSplitter.get_separators_for_language(
    Language.PYTHON
)
print(separators)

['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ', '']


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from pathlib import Path

markdown_text = Path("data/equipment_manual.md").read_text(encoding="utf-8")

